# Light-independent fallback mitigation

The robustness analysis showed that the fitted models share a severe dependency on Light. This notebook asks whether an already-available **Temperature + CO₂ fallback model** can recover useful predictions when a Light fault is known to be active.

The experiment uses an **oracle router**: because we created the simulated fault, we tell the system exactly when to switch. This measures the best-case value of fallback, not whether a real system can detect the fault.

## 1. How the experiment works

```text
Light healthy  -> Temperature + Light + CO₂ primary model
Known Light fault -> Temperature + CO₂ fallback model
```

Only fallback candidates using non-Light sensors already present in the primary system are eligible. Their choice is based on chronological cross-validation inside the training period; Test 1 and Test 2 do not influence selection.

The experiment has three distinct parts:

1. **Selection:** choose a Light-independent fallback using training-only validation results.
2. **Intervention:** create controlled copies of held-out data in which only the Light signal is altered. The original source files are never changed.
3. **Comparison:** measure the faulted primary and the Light-independent fallback on the same held-out observations.

The oracle router knows which intervention was applied because the experiment created it. It does not infer a fault from the sensor readings, so this notebook measures potential mitigation benefit rather than the performance of a real fault-detection system.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Support running from either the repository root or the notebooks folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULT_DIR = PROJECT_ROOT / "models" / "fallback_mitigation"
METRICS_PATH = RESULT_DIR / "fallback_metrics.csv"
CANDIDATES_PATH = RESULT_DIR / "fallback_candidates.csv"

if not METRICS_PATH.exists() or not CANDIDATES_PATH.exists():
    raise FileNotFoundError(
        "Run `python -m sensorbudget.robustness.fallback` before this notebook."
    )

metrics = pd.read_csv(METRICS_PATH)
candidates = pd.read_csv(CANDIDATES_PATH)

PLOTLY_TEMPLATE = "plotly_white"
SPLIT_LABELS = {"test_1": "Test 1", "test_2": "Test 2"}
STRATEGY_LABELS = {
    "primary_clean": "Primary model",
    "fallback_only": "Fallback model",
    "primary_under_fault": "Primary under fault",
    "oracle_gated_fallback": "Oracle-routed fallback",
}
FEATURE_LABELS = {
    "temperature": "Temperature",
    "co2": "CO₂",
    "temperature__co2": "Temperature + CO₂",
}
metrics["split_label"] = metrics["split"].map(SPLIT_LABELS)
metrics["strategy_label"] = metrics["strategy"].map(STRATEGY_LABELS)
candidates["candidate_label"] = candidates["feature_set"].map(FEATURE_LABELS)
print(f"Loaded {len(metrics)} evaluation rows and {len(candidates)} candidates.")

## 2. Training-only fallback selection

Each bar is the mean validation F1 of the already-selected model for that non-Light sensor combination. Higher is better. This section reuses the chronological cross-validation results generated by the earlier sensor-budget training; it does not retrain the models. It filters that existing table to candidates that exclude Light and use only sensors already installed in the primary Temperature + Light + CO₂ system. Test 1 and Test 2 remain untouched during this selection.

In [ ]:
ordered = candidates.sort_values("cv_f1_mean")
colors = ordered["selected_fallback"].map({True: "#2a9d8f", False: "#a8b3bd"})
fig = go.Figure(
    go.Bar(
        x=ordered["cv_f1_mean"],
        y=ordered["candidate_label"],
        orientation="h",
        marker_color=colors,
        customdata=ordered[["selected_model", "selected_fallback"]],
        hovertemplate=(
            "%{y}<br>Mean validation F1: %{x:.3f}"
            "<br>Model: %{customdata[0]}"
            "<br>Selected: %{customdata[1]}<extra></extra>"
        ),
    )
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Fallback selection from chronological training validation",
    xaxis_title="Mean validation F1",
    yaxis_title="Non-Light sensor combination",
    xaxis={"range": [0, 0.75]},
    height=390,
)
fig.show()

**Conclusion.** Temperature + CO₂ is selected because its mean chronological-validation F1 of 0.679 is slightly higher than CO₂ alone and clearly higher than Temperature alone. This selection happens before examining either held-out period.

## 3. Clean-data trade-off

This is the **control experiment**. No sensor values are missing, replaced, frozen, or shifted. The primary and fallback models receive the original Test 1 and Test 2 observations. Comparing them establishes the fallback's clean-data performance ceiling and quantifies the cost of an unnecessary switch when Light is actually healthy.

In [ ]:
clean = metrics.loc[metrics["scenario_group"] == "baseline"].copy()
fig = go.Figure()
# Plotly stacks horizontal grouped traces bottom-to-top. Drawing the
# fallback first therefore places the primary bar above it.
for strategy in ["fallback_only", "primary_clean"]:
    rows = clean.loc[clean["strategy"] == strategy]
    fig.add_trace(
        go.Bar(
            x=rows["f1"],
            y=rows["split_label"],
            orientation="h",
            name=STRATEGY_LABELS[strategy],
            hovertemplate=(
                STRATEGY_LABELS[strategy]
                + "<br>%{y}<br>F1: %{x:.3f}<extra></extra>"
            ),
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Clean held-out performance before routing",
    xaxis_title="F1",
    yaxis_title="Held-out period",
    xaxis={"range": [0, 1.01]},
    barmode="group",
    legend={"traceorder": "reversed"},
    height=390,
)
fig.show()

**Conclusion.** The fallback remains useful but is weaker than the clean primary: F1 is 0.817 versus 0.971 on Test 1 and 0.540 versus 0.980 on Test 2. The large Test 2 gap means switching has a real performance cost and should occur only when the primary input is sufficiently unreliable.

## 4. Severe and policy-related Light faults

The bars compare leaving the altered Light signal with the primary model against oracle routing to the fallback. Temperature, CO₂, timestamps, and occupancy labels remain unchanged in every case. The interventions are applied to copies of the held-out data.

| Scenario | What is changed? | Nature of the failure |
|---|---|---|
| **Complete Light loss** | Every Light reading is treated as unavailable and replaced with the training-period median before the primary model predicts. | A hardware, connection, or ingestion failure with basic median imputation. The median is a fixed fallback value, not a reconstruction of the missing reading. |
| **Light fixed at training 5th percentile (low)** | Every Light reading is replaced by the same low but plausible value calculated from training data. | A silent stuck-sensor failure: numerical readings continue to arrive, but the sensor no longer responds to changing conditions. |
| **Light fixed at training 95th percentile (high)** | Every Light reading is replaced by the same high but plausible training value. | The same silent stuck-sensor failure at the opposite end of the usual range. The 95th percentile is not the absolute maximum. |
| **Unoccupied but lit** | Rows labelled unoccupied receive the median Light value observed for occupied training rows. | A diagnostic policy change, such as lights remaining on in an empty room. This is not necessarily a broken sensor. |
| **Occupied but dark** | Rows labelled occupied receive the median Light value observed for unoccupied training rows. | A diagnostic policy change, such as occupants working with the lights off. It deliberately breaks the historical Light–occupancy shortcut. |

The two lighting-policy scenarios use the known occupancy labels to construct counterfactual test data. They are stress tests for model dependence and are not transformations that could be performed by a live prediction system, where the true occupancy is unknown.

In [ ]:
fault_labels = {
    ("complete_loss", "median_imputation"): "Complete Light loss",
    ("stuck_sensor", "stuck_low"): "Light fixed at training 5th percentile (low)",
    ("stuck_sensor", "stuck_high"): "Light fixed at training 95th percentile (high)",
    ("light_policy", "unoccupied_lit"): "Unoccupied but lit",
    ("light_policy", "occupied_dark"): "Occupied but dark",
}
severe = metrics.loc[
    metrics["scenario_group"].isin(["complete_loss", "stuck_sensor", "light_policy"])
].copy()
severe["fault_label"] = [
    fault_labels[(group, scenario)]
    for group, scenario in zip(severe["scenario_group"], severe["scenario"])
]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Test 1", "Test 2"), shared_yaxes=True)
strategy_colors = {
    "primary_under_fault": "#e76f51",
    "oracle_gated_fallback": "#2a9d8f",
}
for column, split in enumerate(["test_1", "test_2"], start=1):
    split_rows = severe.loc[severe["split"] == split]
    # Draw fallback first so the primary bar appears above it.
    for strategy in ["oracle_gated_fallback", "primary_under_fault"]:
        rows = split_rows.loc[split_rows["strategy"] == strategy]
        value_labels = rows["f1"].map(lambda value: f"{value:.3f}")
        fig.add_trace(
            go.Bar(
                x=rows["f1"],
                y=rows["fault_label"],
                orientation="h",
                name=STRATEGY_LABELS[strategy],
                marker_color=strategy_colors[strategy],
                showlegend=column == 1,
                text=value_labels,
                textposition="outside",
                cliponaxis=False,
                hovertemplate=(
                    STRATEGY_LABELS[strategy]
                    + "<br>%{y}<br>F1: %{x:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
fault_order = list(fault_labels.values())
fig.update_xaxes(range=[0, 1.08], title_text="F1")
fig.update_yaxes(
    categoryorder="array",
    categoryarray=list(reversed(fault_order)),
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Primary model versus oracle-routed fallback under severe Light faults",
    barmode="group",
    legend={"traceorder": "reversed"},
    height=570,
    margin={"l": 285, "r": 35},
)
fig.show()

**Conclusion.** Oracle routing improves F1 in every severe or policy-related condition shown. The largest recovery occurs for complete loss, Light fixed at its low training-percentile value, and occupied darkness, where the primary falls to approximately zero. Recovery is substantial rather than complete because the fallback itself is less accurate and less stable across periods.

## 5. How much F1 does fallback recover?

This section does not introduce new failure simulations. It summarizes the same five interventions from Section 4 as a difference: `fallback F1 - faulted primary F1`. A positive value means the Light-independent fallback performs better than leaving the altered Light signal with the primary model. The value measures potential recovery under perfect fault knowledge; it does not include false alarms or missed detections from a real router.

In [ ]:
gain = severe.loc[severe["strategy"] == "oracle_gated_fallback"].copy()
gain["gain_display"] = gain["mitigation_f1_gain"].map(lambda value: f"{value:+.3f}")
fig = go.Figure()
# Draw Test 2 first so Test 1 appears above it in each horizontal group.
for split in ["test_2", "test_1"]:
    rows = gain.loc[gain["split"] == split]
    fig.add_trace(
        go.Bar(
            x=rows["mitigation_f1_gain"],
            y=rows["fault_label"],
            orientation="h",
            name=SPLIT_LABELS[split],
            customdata=rows[["gain_display", "f1", "faulted_primary_f1"]],
            text=rows["gain_display"],
            textposition="outside",
            cliponaxis=False,
            hovertemplate=(
                "%{y}<br>F1 gain: %{customdata[0]}"
                "<br>Fallback F1: %{customdata[1]:.3f}"
                "<br>Faulted primary F1: %{customdata[2]:.3f}<extra></extra>"
            ),
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="F1 recovered by oracle fallback routing",
    xaxis_title="F1 gain over faulted primary",
    yaxis_title="Simulated Light condition",
    xaxis={"range": [0, 0.9]},
    barmode="group",
    legend={"traceorder": "reversed"},
    height=520,
    margin={"l": 285, "r": 45},
)
fig.update_yaxes(
    categoryorder="array",
    categoryarray=list(reversed(fault_order)),
)
fig.show()

**Conclusion.** The gain ranges from about 0.19 to 0.82 depending on the fault and held-out period. This confirms that fallback can prevent catastrophic failure when the fault is known, but the smaller Test 2 gains also show that mitigation quality is limited by the fallback model's own generalization.

## 6. Linear Light-sensor calibration drift: should we switch models?

This experiment simulates an **additive calibration error in the Light sensor**. The reported Light value begins unchanged, then a positive or negative offset grows linearly through the held-out period until it reaches the stated final size. The actual occupancy labels, Temperature, and CO₂ remain unchanged; this is not a simulation of the room gradually becoming brighter or darker.

For observation `i`, the experiment conceptually calculates:

`altered Light = original Light + progress through period × final bias`

The tested final biases are −1.0, −0.5, +0.5, and +1.0 times the standard deviation of Light calculated from the training period. A positive bias makes the sensor increasingly over-report Light; a negative bias makes it increasingly under-report Light. Unlike the Gaussian-noise experiment in the previous notebook, this is a systematic error that accumulates over time rather than an independent random error at every observation.

The primary model receives the biased Light readings. The fallback line is constant because that model uses only Temperature and CO₂, so changing Light cannot affect its predictions. The comparison asks whether routing the entire held-out period to the fallback would be better at each tested final calibration bias.

In [ ]:
drift = metrics.loc[metrics["scenario_group"] == "gradual_drift"].copy()
drift["severity"] = pd.to_numeric(drift["severity"])
fig = make_subplots(rows=1, cols=2, subplot_titles=("Test 1", "Test 2"), shared_yaxes=True)
for column, split in enumerate(["test_1", "test_2"], start=1):
    split_rows = drift.loc[drift["split"] == split]
    for strategy in ["primary_under_fault", "oracle_gated_fallback"]:
        rows = split_rows.loc[split_rows["strategy"] == strategy].sort_values("severity")
        fig.add_trace(
            go.Scatter(
                x=rows["severity"],
                y=rows["f1"],
                mode="lines+markers",
                name=STRATEGY_LABELS[strategy],
                line={"color": strategy_colors[strategy]},
                showlegend=column == 1,
                hovertemplate=(
                    STRATEGY_LABELS[strategy]
                    + "<br>Final additive bias: %{x:+.1f} std"
                    + "<br>F1: %{y:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
fig.update_yaxes(range=[0, 1.01], title_text="F1", row=1, col=1)
fig.update_xaxes(title_text="Final additive Light bias (training std)")
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Linear additive Light-sensor calibration bias: primary versus fallback",
    height=460,
)
fig.show()

**Conclusion.** The primary remains more accurate than the fallback at every tested linear additive calibration-bias level. Switching merely because a small calibration bias is detected would therefore reduce performance in these cases. A routing rule must consider the estimated severity and expected effect on predictions, not just the presence of sensor bias.

## 7. Overall conclusion — no final sensor recommendation

The oracle experiment establishes that a Light-independent fallback can recover useful predictions during known catastrophic or policy-related Light failures. It also reveals two unresolved problems: fallback performance changes substantially between held-out periods, and unnecessary routing during mild linear Light-sensor calibration bias is harmful.

The next step is not to declare the fallback deployment-ready. It is to create a training-validated fault detector and routing rule, then evaluate false alarms, missed faults, and end-to-end performance without using held-out data for tuning.